# LAB 4 — Dịch máy Anh–Việt bằng Basic Seq2Seq LSTM

> **Phiên bản hướng dẫn dành cho sinh viên mới học**  
> Notebook trình bày đầy đủ quy trình xây dựng một hệ thống dịch máy cơ bản bằng kiến trúc **Encoder–Decoder LSTM** trên dữ liệu PhoMT rút gọn.

## Mục tiêu học tập

Sau bài thực hành, sinh viên có thể:

1. Chuẩn bị dữ liệu song ngữ và xây dựng vocabulary riêng cho ngôn ngữ nguồn, ngôn ngữ đích.
2. Hiểu vai trò của các token đặc biệt `<pad>`, `<bos>`, `<eos>` và `<unk>`.
3. Xây dựng mô hình Encoder–Decoder bằng LSTM.
4. Phân biệt **teacher forcing khi huấn luyện** và **giải mã tự hồi quy khi suy luận**.
5. Huấn luyện, lưu checkpoint, đánh giá và sử dụng mô hình để dịch một câu mới.

---

### Quy ước của bài toán

- Ngôn ngữ nguồn: tiếng Anh (`english`).
- Ngôn ngữ đích: tiếng Việt (`vietnamese`).
- Mỗi tệp dữ liệu là một danh sách JSON gồm các cặp câu song ngữ.
- Vocabulary chỉ được xây dựng từ **train set** để tránh rò rỉ dữ liệu.


## 1. Cài đặt thư viện

Notebook sử dụng:

- `torch`: xây dựng và huấn luyện mô hình Seq2Seq LSTM.
- `nltk`: tính BLEU và chrF ở cấp độ toàn bộ tập dữ liệu.
- Một hàm ROUGE-L được cài đặt trực tiếp trong notebook để sinh viên quan sát rõ cách tính.
- `matplotlib`: trực quan hóa loss.

Chỉ cần chạy ô cài đặt khi môi trường chưa có các thư viện trên.


In [ ]:
%pip install -q nltk matplotlib


## 2. Import, thiết lập seed và chọn thiết bị

### Vì sao cần đặt random seed?

Việc khởi tạo trọng số, xáo trộn dữ liệu và Dropout đều có yếu tố ngẫu nhiên. Đặt seed giúp kết quả giữa các lần chạy ổn định hơn.

> Kết quả vẫn có thể chênh lệch nhẹ giữa CPU, GPU và các phiên bản thư viện khác nhau.


In [ ]:
from collections import Counter
from functools import partial
from pathlib import Path
import json
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.chrf_score import corpus_chrf
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def set_seed(seed: int = 42) -> None:
    """Thiết lập seed cho Python, NumPy và PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Hai tùy chọn dưới ưu tiên tính tái lập hơn tốc độ.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

print(f"PyTorch version: {torch.__version__}")
print(f"Thiết bị đang sử dụng: {DEVICE}")


## 3. Khai báo và kiểm tra đường dẫn dữ liệu

Notebook giả sử thư mục có cấu trúc:

```text
LAB4.ipynb
data/
├── small-train.json
├── small-dev.json
└── small-test.json
```

Mỗi phần tử trong tệp JSON có dạng:

```json
{
  "english": "How are you?",
  "vietnamese": "Bạn khỏe không?"
}
```

Không nên tiếp tục huấn luyện nếu thiếu một trong ba tệp, vì khi đó kết quả đánh giá sẽ không đầy đủ.


In [ ]:
DATA_DIR = Path("data")
TRAIN_PATH = DATA_DIR / "small-train.json"
DEV_PATH = DATA_DIR / "small-dev.json"
TEST_PATH = DATA_DIR / "small-test.json"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "seq2seq_basic_lstm_best.pt"

required_paths = [TRAIN_PATH, DEV_PATH, TEST_PATH]
missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    missing_text = "\n".join(f"- {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Không tìm thấy các tệp dữ liệu sau:\n"
        f"{missing_text}\n\n"
        "Hãy đặt dữ liệu đúng cấu trúc hoặc thay đổi biến DATA_DIR."
    )

print("Đã tìm thấy đầy đủ dữ liệu:")
for path in required_paths:
    print(f"- {path}")


## 4. Đọc và kiểm tra dữ liệu JSON

Trước khi xây dựng mô hình, cần kiểm tra:

- Tệp có chứa một danh sách JSON hay không.
- Mỗi mẫu có đủ hai trường `english` và `vietnamese` hay không.
- Hai câu có đúng kiểu chuỗi và không rỗng hay không.

Kiểm tra dữ liệu sớm giúp phát hiện lỗi định dạng trước khi lỗi xuất hiện sâu trong quá trình huấn luyện.


In [ ]:
SOURCE_FIELD = "english"
TARGET_FIELD = "vietnamese"


def load_parallel_json(path: Path) -> list[dict[str, str]]:
    """Đọc và kiểm tra một tệp dữ liệu song ngữ dạng JSON."""
    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, list):
        raise ValueError(f"{path} phải chứa một danh sách JSON.")

    cleaned_data = []
    for index, item in enumerate(data):
        if not isinstance(item, dict):
            raise ValueError(f"Mẫu {index} trong {path} không phải dictionary.")

        if SOURCE_FIELD not in item or TARGET_FIELD not in item:
            raise KeyError(
                f"Mẫu {index} trong {path} phải có hai trường "
                f"'{SOURCE_FIELD}' và '{TARGET_FIELD}'."
            )

        source_text = item[SOURCE_FIELD]
        target_text = item[TARGET_FIELD]

        if not isinstance(source_text, str) or not isinstance(target_text, str):
            raise TypeError(f"Mẫu {index} trong {path} phải chứa hai chuỗi văn bản.")

        source_text = source_text.strip()
        target_text = target_text.strip()

        if not source_text or not target_text:
            raise ValueError(f"Mẫu {index} trong {path} có câu rỗng.")

        cleaned_data.append(
            {SOURCE_FIELD: source_text, TARGET_FIELD: target_text}
        )

    return cleaned_data


train_data = load_parallel_json(TRAIN_PATH)
dev_data = load_parallel_json(DEV_PATH)
test_data = load_parallel_json(TEST_PATH)

print(f"Train: {len(train_data):,} cặp câu")
print(f"Dev:   {len(dev_data):,} cặp câu")
print(f"Test:  {len(test_data):,} cặp câu")

print("\nVí dụ một cặp câu:")
print("English   :", train_data[0][SOURCE_FIELD])
print("Vietnamese:", train_data[0][TARGET_FIELD])


## 5. Tokenization và xây dựng vocabulary

### Tokenization

Tokenizer cơ bản dưới đây:

1. Chuyển văn bản về chữ thường.
2. Tách từ và giữ lại dấu câu như các token riêng.

Ví dụ:

```text
"How are you?" → ["how", "are", "you", "?"]
```

### Các token đặc biệt

- `<pad>`: thêm vào câu ngắn để các câu trong batch có cùng độ dài.
- `<bos>`: đánh dấu bắt đầu câu đích.
- `<eos>`: đánh dấu kết thúc câu.
- `<unk>`: thay thế token không có trong vocabulary.

Vocabulary **chỉ được xây từ train set**. Nếu đọc cả dev/test để tạo vocabulary, mô hình đã gián tiếp nhìn thấy thông tin của tập đánh giá.


In [ ]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def tokenize(text: str) -> list[str]:
    """Tokenizer đơn giản, dùng chung cho tiếng Anh và tiếng Việt."""
    return TOKEN_PATTERN.findall(text.lower().strip())


def detokenize(tokens: list[str]) -> str:
    """Ghép token thành câu và loại khoảng trắng thừa trước dấu câu."""
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:%\)\]\}])", r"\1", text)
    text = re.sub(r"([\(\[\{])\s+", r"\1", text)
    return text.strip()


class Vocabulary:
    """Ánh xạ hai chiều giữa token và chỉ số nguyên."""

    PAD_TOKEN = "<pad>"
    BOS_TOKEN = "<bos>"
    EOS_TOKEN = "<eos>"
    UNK_TOKEN = "<unk>"

    def __init__(self, min_freq: int = 1) -> None:
        self.min_freq = min_freq
        self.special_tokens = [
            self.PAD_TOKEN,
            self.BOS_TOKEN,
            self.EOS_TOKEN,
            self.UNK_TOKEN,
        ]
        self.itos: list[str] = []
        self.stoi: dict[str, int] = {}

    def build(self, tokenized_sentences: list[list[str]]) -> None:
        counter = Counter(
            token
            for sentence in tokenized_sentences
            for token in sentence
        )

        # Sắp xếp theo tần suất giảm dần rồi theo alphabet để kết quả xác định.
        regular_tokens = [
            token
            for token, frequency in sorted(
                counter.items(), key=lambda item: (-item[1], item[0])
            )
            if frequency >= self.min_freq
            and token not in self.special_tokens
        ]

        self.itos = self.special_tokens + regular_tokens
        self.stoi = {token: index for index, token in enumerate(self.itos)}

    @property
    def pad_idx(self) -> int:
        return self.stoi[self.PAD_TOKEN]

    @property
    def bos_idx(self) -> int:
        return self.stoi[self.BOS_TOKEN]

    @property
    def eos_idx(self) -> int:
        return self.stoi[self.EOS_TOKEN]

    @property
    def unk_idx(self) -> int:
        return self.stoi[self.UNK_TOKEN]

    def __len__(self) -> int:
        return len(self.itos)

    def encode(self, text: str, add_special_tokens: bool = True) -> list[int]:
        token_ids = [
            self.stoi.get(token, self.unk_idx)
            for token in tokenize(text)
        ]

        if add_special_tokens:
            return [self.bos_idx, *token_ids, self.eos_idx]
        return token_ids

    def decode(self, token_ids, stop_at_eos: bool = True) -> str:
        tokens = []
        for token_id in token_ids:
            token_id = int(token_id)

            if stop_at_eos and token_id == self.eos_idx:
                break

            if token_id in {self.pad_idx, self.bos_idx}:
                continue

            token = self.itos[token_id] if token_id < len(self.itos) else self.UNK_TOKEN
            tokens.append(token)

        return detokenize(tokens)

    def to_dict(self) -> dict:
        return {"min_freq": self.min_freq, "itos": self.itos}

    @classmethod
    def from_dict(cls, state: dict):
        vocabulary = cls(min_freq=state["min_freq"])
        vocabulary.itos = list(state["itos"])
        vocabulary.stoi = {
            token: index for index, token in enumerate(vocabulary.itos)
        }
        return vocabulary


MIN_FREQUENCY = 1

source_vocab = Vocabulary(min_freq=MIN_FREQUENCY)
target_vocab = Vocabulary(min_freq=MIN_FREQUENCY)

source_vocab.build([
    tokenize(item[SOURCE_FIELD]) for item in train_data
])
target_vocab.build([
    tokenize(item[TARGET_FIELD]) for item in train_data
])

print(f"Kích thước vocabulary tiếng Anh : {len(source_vocab):,}")
print(f"Kích thước vocabulary tiếng Việt: {len(target_vocab):,}")
print("Các token đặc biệt:", target_vocab.itos[:4])


## 6. Kiểm tra mã hóa và giải mã một câu

Trước khi tạo `Dataset`, nên kiểm tra thủ công một ví dụ:

- Câu được tách thành các token nào?
- Chỉ số đầu tiên có phải `<bos>` hay không?
- Chỉ số cuối cùng có phải `<eos>` hay không?
- Giải mã lại có gần với câu ban đầu hay không?

Đây là bước đơn giản nhưng giúp phát hiện nhiều lỗi vocabulary và tokenizer.


In [ ]:
example_source = train_data[0][SOURCE_FIELD]
example_target = train_data[0][TARGET_FIELD]

source_ids = source_vocab.encode(example_source)
target_ids = target_vocab.encode(example_target)

print("Câu nguồn       :", example_source)
print("Token nguồn     :", tokenize(example_source))
print("Chỉ số nguồn    :", source_ids)
print("Giải mã nguồn   :", source_vocab.decode(source_ids))

print("\nCâu đích        :", example_target)
print("Token đích      :", tokenize(example_target))
print("Chỉ số đích     :", target_ids)
print("Giải mã đích    :", target_vocab.decode(target_ids))

assert source_ids[0] == source_vocab.bos_idx
assert source_ids[-1] == source_vocab.eos_idx
assert target_ids[0] == target_vocab.bos_idx
assert target_ids[-1] == target_vocab.eos_idx


## 7. Xây dựng `Dataset`

`TranslationDataset` chuyển mỗi cặp câu thành hai tensor số nguyên:

```text
source_text → source_ids
target_text → target_ids
```

Dữ liệu được mã hóa một lần trong hàm khởi tạo để giảm công việc lặp lại trong quá trình huấn luyện.


In [ ]:
class TranslationDataset(Dataset):
    def __init__(
        self,
        data: list[dict[str, str]],
        source_vocab: Vocabulary,
        target_vocab: Vocabulary,
    ) -> None:
        self.samples = []

        for item in data:
            source_ids = torch.tensor(
                source_vocab.encode(item[SOURCE_FIELD]),
                dtype=torch.long,
            )
            target_ids = torch.tensor(
                target_vocab.encode(item[TARGET_FIELD]),
                dtype=torch.long,
            )

            self.samples.append(
                {
                    "source_ids": source_ids,
                    "target_ids": target_ids,
                    "source_text": item[SOURCE_FIELD],
                    "target_text": item[TARGET_FIELD],
                }
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict:
        return self.samples[index]


train_dataset = TranslationDataset(train_data, source_vocab, target_vocab)
dev_dataset = TranslationDataset(dev_data, source_vocab, target_vocab)
test_dataset = TranslationDataset(test_data, source_vocab, target_vocab)

sample = train_dataset[0]
print("source_ids:", sample["source_ids"].shape, sample["source_ids"][:10])
print("target_ids:", sample["target_ids"].shape, sample["target_ids"][:10])


## 8. Tạo batch có padding bằng `collate_fn`

Các câu trong cùng một batch thường có độ dài khác nhau. `pad_sequence` bổ sung `<pad>` vào cuối câu ngắn.

Ngoài tensor đã padding, encoder còn cần **độ dài thật** của từng câu. Độ dài này được sử dụng bởi `pack_padded_sequence` để LSTM không xem padding là nội dung của câu.

Ví dụ:

```text
[<bos>, i, agree, <eos>]
[<bos>, thank, you, very, much, <eos>]
```

sau padding trở thành:

```text
[<bos>, i, agree, <eos>, <pad>, <pad>]
[<bos>, thank, you, very, much, <eos>]
```


In [ ]:
def collate_translation_batch(
    batch: list[dict],
    source_pad_idx: int,
    target_pad_idx: int,
) -> dict[str, torch.Tensor | list[str]]:
    source_sequences = [item["source_ids"] for item in batch]
    target_sequences = [item["target_ids"] for item in batch]

    source_lengths = torch.tensor(
        [len(sequence) for sequence in source_sequences],
        dtype=torch.long,
    )

    padded_sources = pad_sequence(
        source_sequences,
        batch_first=True,
        padding_value=source_pad_idx,
    )
    padded_targets = pad_sequence(
        target_sequences,
        batch_first=True,
        padding_value=target_pad_idx,
    )

    return {
        "source_ids": padded_sources,
        "source_lengths": source_lengths,
        "target_ids": padded_targets,
        "source_texts": [item["source_text"] for item in batch],
        "target_texts": [item["target_text"] for item in batch],
    }


collate_fn = partial(
    collate_translation_batch,
    source_pad_idx=source_vocab.pad_idx,
    target_pad_idx=target_vocab.pad_idx,
)

example_batch = collate_fn([train_dataset[0], train_dataset[1]])
print("source_ids shape    :", example_batch["source_ids"].shape)
print("source_lengths      :", example_batch["source_lengths"])
print("target_ids shape    :", example_batch["target_ids"].shape)


## 9. Tạo `DataLoader`

- Train loader dùng `shuffle=True` để thay đổi thứ tự mẫu sau mỗi epoch.
- Dev và test loader dùng `shuffle=False` để kết quả có thứ tự ổn định.
- `pin_memory=True` có thể tăng tốc chuyển dữ liệu từ CPU sang GPU.
- `num_workers=0` tương thích tốt với nhiều môi trường notebook; có thể tăng nếu máy hỗ trợ.


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

batch = next(iter(train_loader))
print("Batch source:", batch["source_ids"].shape)
print("Batch target:", batch["target_ids"].shape)
print("Độ dài thật  :", batch["source_lengths"][:10].tolist())


## 10. Khảo sát độ dài câu

Độ dài câu ảnh hưởng trực tiếp đến:

- lượng bộ nhớ cần dùng;
- thời gian huấn luyện;
- độ khó của mô hình Seq2Seq không có Attention;
- giá trị `MAX_DECODE_LENGTH` khi dịch.

Trong mô hình thực tế, có thể giới hạn hoặc gom các câu có độ dài gần nhau vào cùng batch. Bài này giữ toàn bộ dữ liệu để tập trung vào kiến trúc cơ bản.


In [ ]:
def describe_lengths(data: list[dict[str, str]], field: str) -> dict[str, float]:
    lengths = np.array([len(tokenize(item[field])) + 2 for item in data])
    return {
        "min": int(lengths.min()),
        "mean": float(lengths.mean()),
        "median": float(np.median(lengths)),
        "p95": float(np.percentile(lengths, 95)),
        "max": int(lengths.max()),
    }


source_stats = describe_lengths(train_data, SOURCE_FIELD)
target_stats = describe_lengths(train_data, TARGET_FIELD)

print("Độ dài câu tiếng Anh :", source_stats)
print("Độ dài câu tiếng Việt:", target_stats)

# Thêm một khoảng dự phòng so với phân vị 95% của câu đích.
MAX_DECODE_LENGTH = max(10, int(target_stats["p95"]) + 10)
print(f"MAX_DECODE_LENGTH = {MAX_DECODE_LENGTH}")


## 11. Lý thuyết Basic Seq2Seq LSTM

Mô hình gồm hai phần.

### Encoder

Encoder đọc toàn bộ câu tiếng Anh và trả về trạng thái cuối:

- hidden state `h`;
- cell state `c`.

Hai trạng thái này được xem như biểu diễn nén của câu nguồn.

### Decoder

Decoder nhận `(h, c)` từ encoder và sinh câu tiếng Việt từng token một.

### Khi huấn luyện: teacher forcing

Decoder nhận chuỗi đích đã dịch phải một vị trí:

```text
Input decoder : <bos> tôi đồng ý
Nhãn cần đoán : tôi đồng ý <eos>
```

### Khi suy luận: autoregressive decoding

Không có câu tiếng Việt đúng. Token vừa dự đoán được đưa lại vào decoder để dự đoán token tiếp theo, cho đến khi gặp `<eos>` hoặc đạt độ dài tối đa.

> Nếu đánh giá bằng cách đưa câu đích đúng vào decoder, mô hình đã được cung cấp thông tin đáp án. Điểm số đó không phản ánh khả năng dịch thực tế.

### Hạn chế của kiến trúc cơ bản

Toàn bộ câu nguồn phải được nén vào `(h, c)`. Với câu dài, đây là nút thắt thông tin. Attention và Transformer được phát triển để khắc phục hạn chế này.


In [ ]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        pad_idx: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        embedded = self.dropout(self.embedding(source_ids))

        packed = pack_padded_sequence(
            embedded,
            lengths=source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, (hidden, cell) = self.lstm(packed)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        pad_idx: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.output_layer = nn.Linear(hidden_size, vocab_size)

    def forward(
        self,
        decoder_input_ids: torch.Tensor,
        hidden: torch.Tensor,
        cell: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        embedded = self.dropout(self.embedding(decoder_input_ids))
        outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        logits = self.output_layer(outputs)
        return logits, hidden, cell


class Seq2SeqLSTM(nn.Module):
    def __init__(
        self,
        source_vocab_size: int,
        target_vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        source_pad_idx: int,
        target_pad_idx: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.encoder = Encoder(
            vocab_size=source_vocab_size,
            embedding_dim=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            pad_idx=source_pad_idx,
            dropout=dropout,
        )
        self.decoder = Decoder(
            vocab_size=target_vocab_size,
            embedding_dim=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            pad_idx=target_pad_idx,
            dropout=dropout,
        )

    def forward(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
        target_ids: torch.Tensor,
    ) -> torch.Tensor:
        """Forward khi huấn luyện với teacher forcing 100%."""
        hidden, cell = self.encoder(source_ids, source_lengths)

        # Bỏ token cuối; mỗi vị trí dự đoán token kế tiếp.
        decoder_inputs = target_ids[:, :-1]
        logits, _, _ = self.decoder(decoder_inputs, hidden, cell)
        return logits

    @torch.inference_mode()
    def greedy_decode(
        self,
        source_ids: torch.Tensor,
        source_lengths: torch.Tensor,
        bos_idx: int,
        eos_idx: int,
        max_length: int,
    ) -> torch.Tensor:
        """Sinh bản dịch chỉ từ câu nguồn, không dùng câu đích tham chiếu."""
        hidden, cell = self.encoder(source_ids, source_lengths)

        batch_size = source_ids.size(0)
        current_tokens = torch.full(
            (batch_size, 1),
            fill_value=bos_idx,
            dtype=torch.long,
            device=source_ids.device,
        )

        generated_tokens = []
        finished = torch.zeros(
            batch_size,
            dtype=torch.bool,
            device=source_ids.device,
        )

        for _ in range(max_length):
            logits, hidden, cell = self.decoder(
                current_tokens,
                hidden,
                cell,
            )
            next_tokens = logits[:, -1].argmax(dim=-1)
            generated_tokens.append(next_tokens)

            finished |= next_tokens.eq(eos_idx)
            if finished.all():
                break

            # Những câu đã kết thúc tiếp tục nhận EOS để không sinh nội dung mới.
            next_tokens = torch.where(
                finished,
                torch.full_like(next_tokens, eos_idx),
                next_tokens,
            )
            current_tokens = next_tokens.unsqueeze(1)

        if not generated_tokens:
            return torch.empty(
                (batch_size, 0),
                dtype=torch.long,
                device=source_ids.device,
            )

        return torch.stack(generated_tokens, dim=1)


## 12. Chọn mô hình và siêu tham số

Các siêu tham số dưới đây là cấu hình khởi đầu, không phải giá trị tối ưu cho mọi tập dữ liệu.

- `EMBEDDING_DIM`: số chiều vector biểu diễn token.
- `HIDDEN_SIZE`: kích thước hidden state và cell state.
- `NUM_LAYERS`: số tầng LSTM của encoder và decoder.
- `DROPOUT`: giúp giảm overfitting.

Encoder và decoder phải có cùng `HIDDEN_SIZE` và `NUM_LAYERS` vì trạng thái cuối của encoder được chuyển trực tiếp sang decoder.


In [ ]:
MODEL_CONFIG = {
    "embedding_dim": 256,
    "hidden_size": 256,
    "num_layers": 2,
    "dropout": 0.30,
}

model = Seq2SeqLSTM(
    source_vocab_size=len(source_vocab),
    target_vocab_size=len(target_vocab),
    source_pad_idx=source_vocab.pad_idx,
    target_pad_idx=target_vocab.pad_idx,
    **MODEL_CONFIG,
).to(DEVICE)


def count_trainable_parameters(module: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


print(model)
print(f"\nSố tham số có thể học: {count_trainable_parameters(model):,}")


## 13. Loss function, optimizer và gradient clipping

### Cross-Entropy Loss

Tại mỗi vị trí, mô hình dự đoán xác suất trên toàn bộ vocabulary tiếng Việt. Cross-Entropy đo mức khác biệt giữa phân phối dự đoán và token đúng.

Token `<pad>` không phải nội dung thật nên được bỏ qua bằng `ignore_index`.

### Loss theo token

Các câu có độ dài khác nhau. Vì vậy, loss theo dõi được tính bằng:

```text
tổng loss của các token thật / tổng số token thật
```

### Gradient clipping

LSTM có thể gặp exploding gradient. `clip_grad_norm_` giới hạn norm của gradient trước khi optimizer cập nhật trọng số.


In [ ]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
GRADIENT_CLIP = 1.0
NUM_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3

criterion = nn.CrossEntropyLoss(
    ignore_index=target_vocab.pad_idx,
    reduction="sum",
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(optimizer)


## 14. Hàm huấn luyện một epoch

Trong mỗi batch:

1. Chuyển tensor sang thiết bị.
2. Chạy forward với teacher forcing.
3. Dịch nhãn sang trái một vị trí: `target[:, 1:]`.
4. Tính loss trên các token không phải `<pad>`.
5. Lan truyền ngược, clipping gradient và cập nhật trọng số.

`optimizer.zero_grad(set_to_none=True)` thường tiết kiệm bộ nhớ hơn việc đặt toàn bộ gradient về 0.


In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    target_pad_idx: int,
    gradient_clip: float,
) -> float:
    model.train()

    total_loss_sum = 0.0
    total_target_tokens = 0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for batch in progress_bar:
        source_ids = batch["source_ids"].to(DEVICE, non_blocking=True)
        source_lengths = batch["source_lengths"]
        target_ids = batch["target_ids"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(source_ids, source_lengths, target_ids)
        expected_tokens = target_ids[:, 1:]

        loss_sum = criterion(
            logits.reshape(-1, logits.size(-1)),
            expected_tokens.reshape(-1),
        )

        token_count = expected_tokens.ne(target_pad_idx).sum()
        loss = loss_sum / token_count.clamp_min(1)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        optimizer.step()

        total_loss_sum += loss_sum.item()
        total_target_tokens += token_count.item()

        progress_bar.set_postfix(
            token_loss=f"{total_loss_sum / max(total_target_tokens, 1):.4f}"
        )

    return total_loss_sum / max(total_target_tokens, 1)


## 15. Hàm đánh giá loss và chất lượng bản dịch

Notebook tách hai kiểu đánh giá:

### Dev loss

Dev loss vẫn dùng teacher forcing, phù hợp để theo dõi quá trình tối ưu và early stopping.

### BLEU, chrF và ROUGE-L

Các chỉ số chất lượng dịch phải dùng `greedy_decode`, tức là mô hình chỉ nhìn thấy câu tiếng Anh.

- **BLEU**: dựa trên mức trùng khớp n-gram.
- **chrF**: dựa trên n-gram ký tự, thường hữu ích khi cách tách từ chưa hoàn hảo.
- **ROUGE-L**: dựa trên dãy con chung dài nhất.

Các chỉ số tự động không thể đánh giá đầy đủ độ đúng nghĩa và độ tự nhiên. Luôn cần xem thêm các ví dụ dịch cụ thể.


In [ ]:
@torch.inference_mode()
def evaluate_token_loss(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    target_pad_idx: int,
) -> float:
    model.eval()

    total_loss_sum = 0.0
    total_target_tokens = 0

    for batch in tqdm(loader, desc="Evaluating loss", leave=False):
        source_ids = batch["source_ids"].to(DEVICE, non_blocking=True)
        source_lengths = batch["source_lengths"]
        target_ids = batch["target_ids"].to(DEVICE, non_blocking=True)

        logits = model(source_ids, source_lengths, target_ids)
        expected_tokens = target_ids[:, 1:]

        loss_sum = criterion(
            logits.reshape(-1, logits.size(-1)),
            expected_tokens.reshape(-1),
        )
        token_count = expected_tokens.ne(target_pad_idx).sum()

        total_loss_sum += loss_sum.item()
        total_target_tokens += token_count.item()

    return total_loss_sum / max(total_target_tokens, 1)


@torch.inference_mode()
def generate_corpus_predictions(
    model: Seq2SeqLSTM,
    loader: DataLoader,
    target_vocab: Vocabulary,
    max_decode_length: int,
) -> tuple[list[str], list[str], list[str]]:
    model.eval()

    sources = []
    references = []
    predictions = []

    for batch in tqdm(loader, desc="Generating translations", leave=False):
        source_ids = batch["source_ids"].to(DEVICE, non_blocking=True)
        source_lengths = batch["source_lengths"]

        predicted_ids = model.greedy_decode(
            source_ids=source_ids,
            source_lengths=source_lengths,
            bos_idx=target_vocab.bos_idx,
            eos_idx=target_vocab.eos_idx,
            max_length=max_decode_length,
        )

        sources.extend(batch["source_texts"])
        references.extend(batch["target_texts"])
        predictions.extend(
            target_vocab.decode(sequence.cpu().tolist())
            for sequence in predicted_ids
        )

    return sources, references, predictions


def longest_common_subsequence_length(
    reference_tokens: list[str],
    prediction_tokens: list[str],
) -> int:
    """Tính độ dài dãy con chung dài nhất bằng quy hoạch động."""
    if not reference_tokens or not prediction_tokens:
        return 0

    previous_row = [0] * (len(prediction_tokens) + 1)

    for reference_token in reference_tokens:
        current_row = [0]
        for column, prediction_token in enumerate(prediction_tokens, start=1):
            if reference_token == prediction_token:
                current_row.append(previous_row[column - 1] + 1)
            else:
                current_row.append(max(previous_row[column], current_row[-1]))
        previous_row = current_row

    return previous_row[-1]


def rouge_l_f1(reference: str, prediction: str) -> float:
    """Tính ROUGE-L F1 trên token; kết quả nằm trong khoảng [0, 1]."""
    reference_tokens = tokenize(reference)
    prediction_tokens = tokenize(prediction)

    if not reference_tokens or not prediction_tokens:
        return 0.0

    lcs_length = longest_common_subsequence_length(
        reference_tokens,
        prediction_tokens,
    )
    precision = lcs_length / len(prediction_tokens)
    recall = lcs_length / len(reference_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def compute_translation_metrics(
    references: list[str],
    predictions: list[str],
) -> dict[str, float]:
    if len(references) != len(predictions):
        raise ValueError("Số câu tham chiếu và dự đoán phải bằng nhau.")

    tokenized_references = [[tokenize(reference)] for reference in references]
    tokenized_predictions = [tokenize(prediction) for prediction in predictions]

    smoothing = SmoothingFunction().method1
    bleu = 100 * corpus_bleu(
        tokenized_references,
        tokenized_predictions,
        smoothing_function=smoothing,
    )

    # NLTK chrF nhận danh sách chuỗi tham chiếu và dự đoán.
    chrf = 100 * corpus_chrf(references, predictions)

    rouge_scores = [
        rouge_l_f1(reference, prediction)
        for reference, prediction in zip(references, predictions)
    ]
    rouge_l = 100 * float(np.mean(rouge_scores)) if rouge_scores else 0.0

    return {
        "BLEU": bleu,
        "chrF": chrf,
        "ROUGE-L": rouge_l,
    }


## 16. Huấn luyện và lưu checkpoint tốt nhất

Mô hình được lưu khi **dev loss giảm**.

Checkpoint gồm:

- trọng số mô hình;
- trạng thái optimizer;
- cấu hình mô hình;
- vocabulary nguồn và đích;
- epoch và dev loss tốt nhất.

Early stopping dừng quá trình nếu dev loss không cải thiện trong một số epoch liên tiếp. Điều này giúp hạn chế overfitting và tiết kiệm thời gian.


In [ ]:
history = {
    "train_loss": [],
    "dev_loss": [],
}

best_dev_loss = float("inf")
epochs_without_improvement = 0
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        target_pad_idx=target_vocab.pad_idx,
        gradient_clip=GRADIENT_CLIP,
    )

    dev_loss = evaluate_token_loss(
        model=model,
        loader=dev_loader,
        criterion=criterion,
        target_pad_idx=target_vocab.pad_idx,
    )

    history["train_loss"].append(train_loss)
    history["dev_loss"].append(dev_loss)

    improved = dev_loss < best_dev_loss
    marker = " ← lưu checkpoint" if improved else ""

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train token loss: {train_loss:.4f} | "
        f"Dev token loss: {dev_loss:.4f}{marker}"
    )

    if improved:
        best_dev_loss = dev_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "best_dev_loss": best_dev_loss,
                "model_config": MODEL_CONFIG,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "source_vocab": source_vocab.to_dict(),
                "target_vocab": target_vocab.to_dict(),
                "history": history,
                "max_decode_length": MAX_DECODE_LENGTH,
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping: dev loss không cải thiện trong "
            f"{EARLY_STOPPING_PATIENCE} epoch liên tiếp."
        )
        break

elapsed = time.time() - training_start
print(f"\nThời gian huấn luyện: {elapsed / 60:.2f} phút")
print(f"Checkpoint tốt nhất: {CHECKPOINT_PATH}")
print(f"Best dev token loss: {best_dev_loss:.4f}")


## 17. Vẽ biểu đồ quá trình học

Biểu đồ train loss và dev loss giúp nhận biết:

- cả hai cùng giảm: mô hình đang học;
- train loss giảm nhưng dev loss tăng: dấu hiệu overfitting;
- cả hai gần như không giảm: cần xem lại learning rate, dữ liệu hoặc kiến trúc.

Không nên chỉ nhìn train loss để kết luận mô hình tốt.


In [ ]:
epoch_axis = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epoch_axis, history["train_loss"], marker="o", label="Train loss")
plt.plot(epoch_axis, history["dev_loss"], marker="o", label="Dev loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss trên token")
plt.title("Quá trình huấn luyện Basic Seq2Seq LSTM")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 18. Tải lại checkpoint tốt nhất

Không nên dùng trọng số của epoch cuối một cách mặc định. Epoch tốt nhất trên dev set có thể xuất hiện sớm hơn.

Vocabulary cũng được tải từ checkpoint để bảo đảm chỉ số token hoàn toàn khớp với embedding đã huấn luyện.


In [ ]:
def load_checkpoint(path: Path, device: torch.device) -> dict:
    # Tương thích với cả phiên bản PyTorch có và chưa có tham số weights_only.
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


checkpoint = load_checkpoint(CHECKPOINT_PATH, DEVICE)

loaded_source_vocab = Vocabulary.from_dict(checkpoint["source_vocab"])
loaded_target_vocab = Vocabulary.from_dict(checkpoint["target_vocab"])

best_model = Seq2SeqLSTM(
    source_vocab_size=len(loaded_source_vocab),
    target_vocab_size=len(loaded_target_vocab),
    source_pad_idx=loaded_source_vocab.pad_idx,
    target_pad_idx=loaded_target_vocab.pad_idx,
    **checkpoint["model_config"],
).to(DEVICE)

best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

print(f"Đã tải checkpoint tại epoch {checkpoint['epoch']}")
print(f"Dev token loss tốt nhất: {checkpoint['best_dev_loss']:.4f}")


## 19. Đánh giá cuối cùng trên test set

Test set chỉ được dùng sau khi đã chọn mô hình bằng dev set.

Quy trình đúng:

```text
Train set → học tham số
Dev set   → chọn checkpoint và siêu tham số
Test set  → báo cáo kết quả cuối cùng
```

Không nên điều chỉnh mô hình nhiều lần dựa trên test set vì khi đó test set không còn là dữ liệu chưa quan sát.


In [ ]:
test_token_loss = evaluate_token_loss(
    model=best_model,
    loader=test_loader,
    criterion=criterion,
    target_pad_idx=loaded_target_vocab.pad_idx,
)

sources, references, predictions = generate_corpus_predictions(
    model=best_model,
    loader=test_loader,
    target_vocab=loaded_target_vocab,
    max_decode_length=checkpoint["max_decode_length"],
)

test_metrics = compute_translation_metrics(references, predictions)

print(f"Test token loss: {test_token_loss:.4f}")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name:8s}: {metric_value:.2f}")


## 20. Xem một số bản dịch

Chỉ số tổng hợp không cho biết mô hình sai như thế nào. Cần xem trực tiếp:

- câu nguồn;
- câu tham chiếu;
- câu mô hình sinh ra.

Các lỗi thường gặp của Basic Seq2Seq:

- bỏ sót từ;
- lặp từ;
- kết thúc quá sớm;
- dịch kém khi câu dài;
- sinh nhiều `<unk>` nếu vocabulary nhỏ.


In [ ]:
def show_translation_examples(
    sources: list[str],
    references: list[str],
    predictions: list[str],
    number_of_examples: int = 10,
    seed: int = 42,
) -> None:
    rng = random.Random(seed)
    indices = list(range(len(sources)))
    rng.shuffle(indices)

    for order, index in enumerate(indices[:number_of_examples], start=1):
        print(f"Ví dụ {order}")
        print("EN  :", sources[index])
        print("REF :", references[index])
        print("PRED:", predictions[index] or "<câu rỗng>")
        print("-" * 80)


show_translation_examples(
    sources,
    references,
    predictions,
    number_of_examples=min(10, len(sources)),
)


## 21. Dịch một câu mới

Hàm `translate_sentence` thực hiện đúng quy trình suy luận:

1. Tokenize và mã hóa câu tiếng Anh.
2. Chạy encoder một lần.
3. Bắt đầu decoder bằng `<bos>`.
4. Chọn token xác suất cao nhất ở mỗi bước.
5. Dừng khi gặp `<eos>` hoặc đạt độ dài tối đa.

Đây là **greedy decoding**. Beam search có thể cho kết quả tốt hơn nhưng phức tạp hơn và không thuộc phạm vi bài cơ bản.


In [ ]:
@torch.inference_mode()
def translate_sentence(
    sentence: str,
    model: Seq2SeqLSTM,
    source_vocab: Vocabulary,
    target_vocab: Vocabulary,
    max_decode_length: int,
) -> str:
    model.eval()

    source_tensor = torch.tensor(
        source_vocab.encode(sentence),
        dtype=torch.long,
    ).unsqueeze(0)

    source_length = torch.tensor([source_tensor.size(1)], dtype=torch.long)
    source_tensor = source_tensor.to(DEVICE)

    predicted_ids = model.greedy_decode(
        source_ids=source_tensor,
        source_lengths=source_length,
        bos_idx=target_vocab.bos_idx,
        eos_idx=target_vocab.eos_idx,
        max_length=max_decode_length,
    )

    return target_vocab.decode(predicted_ids[0].cpu().tolist())


new_sentences = [
    "How are you?",
    "Thank you very much.",
    "I love learning natural language processing.",
]

for sentence in new_sentences:
    translation = translate_sentence(
        sentence=sentence,
        model=best_model,
        source_vocab=loaded_source_vocab,
        target_vocab=loaded_target_vocab,
        max_decode_length=checkpoint["max_decode_length"],
    )
    print(f"EN: {sentence}")
    print(f"VI: {translation or '<câu rỗng>'}")
    print("-" * 60)


## 22. Phân tích hạn chế và hướng cải tiến

Basic Seq2Seq LSTM là mô hình nền tảng, nhưng có nhiều hạn chế:

### 1. Nút thắt context vector

Encoder phải nén toàn bộ câu nguồn vào hidden state và cell state cuối. Câu càng dài, nguy cơ mất thông tin càng lớn.

**Cải tiến:** thêm Attention để decoder truy cập trực tiếp các hidden state của encoder.

### 2. Greedy decoding

Chọn token tốt nhất tại từng bước chưa chắc tạo ra câu tốt nhất toàn cục.

**Cải tiến:** dùng Beam Search.

### 3. Vocabulary theo từ

Từ hiếm và từ mới trở thành `<unk>`, làm mất thông tin.

**Cải tiến:** dùng subword tokenization như BPE hoặc SentencePiece.

### 4. Tokenizer đơn giản

Tokenizer bằng biểu thức chính quy chưa xử lý đầy đủ tên riêng, số, viết tắt và các trường hợp đặc biệt.

**Cải tiến:** dùng tokenizer phù hợp với dữ liệu và kiểm tra chất lượng tiền xử lý.

### 5. Đánh giá tự động

BLEU, chrF và ROUGE-L không hoàn toàn phản ánh độ đúng nghĩa.

**Cải tiến:** kết hợp đánh giá con người và các metric ngữ nghĩa hiện đại khi phù hợp.


In [ ]:
# So sánh nhanh độ dài câu và chất lượng dự đoán để gợi ý phân tích lỗi.
length_error_rows = []

for source, reference, prediction in zip(sources, references, predictions):
    length_error_rows.append(
        {
            "source_length": len(tokenize(source)),
            "target_length": len(tokenize(reference)),
            "prediction_length": len(tokenize(prediction)),
            "rouge_l": rouge_l_f1(reference, prediction),
        }
    )

if length_error_rows:
    sorted_rows = sorted(length_error_rows, key=lambda row: row["rouge_l"])
    print("Năm mẫu có ROUGE-L thấp nhất:")
    for row in sorted_rows[:5]:
        print(row)


## 23. Tổng kết kiến thức

Trong bài thực hành này, sinh viên đã hoàn thành toàn bộ pipeline dịch máy cơ bản:

```text
Đọc dữ liệu song ngữ
        ↓
Tokenization
        ↓
Vocabulary chỉ từ train set
        ↓
Dataset + padding + độ dài thật
        ↓
Encoder LSTM
        ↓
Hidden state và cell state
        ↓
Decoder LSTM với teacher forcing
        ↓
Early stopping + checkpoint
        ↓
Autoregressive greedy decoding
        ↓
BLEU + chrF + ROUGE-L + phân tích ví dụ
```

### Những điểm cần ghi nhớ

1. Vocabulary không được xây từ dev/test.
2. Encoder cần bỏ qua padding bằng độ dài thật của câu.
3. Teacher forcing chỉ phù hợp cho huấn luyện và tính dev loss.
4. Đánh giá dịch phải sinh câu mà không dùng câu đích đúng.
5. Checkpoint tốt nhất nên được chọn trên dev set trước khi đánh giá test set.
6. Basic Seq2Seq là nền tảng để học Attention, BiLSTM Encoder và Transformer.

### Bài tập mở rộng

1. Thay greedy decoding bằng Beam Search.
2. Thêm Attention vào decoder và so sánh với mô hình cơ bản.
3. Thử embedding/hidden size khác nhau và báo cáo số tham số.
4. Thử giới hạn vocabulary bằng `MIN_FREQUENCY = 2` hoặc `3`.
5. Phân tích BLEU theo các nhóm độ dài câu.


In [ ]:
print("Hoàn thành LAB 4 — Bài 1: Basic Seq2Seq LSTM.")
print("Bước tiếp theo đề xuất: Seq2Seq LSTM với Attention.")
